# 📘 FULL PIPELINE NOTEBOOK — GPT‑4o Vision OCR → OpenAI Embeddings → Azure ML → GPT‑4o Reasoning
### Updated with FINAL PIPELINE OUTPUT (OCR + Embedding + Score + GPT‑4o Summary)


In [ ]:
!pip install openai pandas tqdm pyarrow scikit-learn joblib requests

In [ ]:
from openai import OpenAI
import pandas as pd, numpy as np, time, os, joblib, requests
from tqdm import tqdm

OPENAI_API_KEY = 'YOUR_OPENAI_KEY'
client = OpenAI(api_key=OPENAI_API_KEY)

## Step 1 — Load Dataset

In [ ]:
sas_url = 'https://<yourstorage>.blob.core.windows.net/multifinben/raw/train-00000-of-00008.parquet?<sas>'
df = pd.read_parquet(sas_url)
df.head()

## Step 2 — GPT‑4o Vision OCR

In [ ]:
def gpt4o_vision_ocr(base64_str):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{
            'role': 'user',
            'content': [
                {'type': 'text', 'text': 'Extract ONLY the text from this image.'},
                {'type': 'image_url', 'image_url': {'url': f'data:image/png;base64,{base64_str}'}}
            ]
        }]
    )
    return response.choices[0].message.content

## Step 3 — Budget‑Safe OCR Loop

In [ ]:
MAX_SPEND_USD = 8.0
COST_PER_IMAGE = 0.005
OUTPUT = 'ocr_budget_safe.pkl'

if 'ocr_text' not in df.columns: df['ocr_text'] = None
if os.path.exists(OUTPUT): df.update(pd.read_pickle(OUTPUT))

spent, processed = 0.0, 0

for i in tqdm(range(len(df))):
    if isinstance(df.loc[i,'ocr_text'], str) and len(df.loc[i,'ocr_text'])>1: continue
    if spent + COST_PER_IMAGE > MAX_SPEND_USD: break
    try:
        text = gpt4o_vision_ocr(df.loc[i,'image'])
        df.loc[i,'ocr_text'] = text
        spent += COST_PER_IMAGE
        processed += 1
        if processed % 20 == 0: df.to_pickle(OUTPUT)
        time.sleep(0.3)
    except Exception as e:
        print('Error:', e); time.sleep(1)

df.to_pickle(OUTPUT)
print('OCR DONE:', processed, 'images | Cost ≈ $', spent)

## Step 4 — OpenAI Embeddings (text-embedding-3-large)

In [ ]:
def embed_text(text):
    if text is None or len(text.strip()) == 0: return None
    response = client.embeddings.create(model='text-embedding-3-large', input=text)
    return response.data[0].embedding

In [ ]:
df['embedding'] = [embed_text(t) for t in tqdm(df['ocr_text'])]
df.to_pickle('processed_embeddings.pkl')
df.head()

## Step 5 — FINAL PIPELINE OUTPUT (OCR + Embedding + Score + GPT‑4o Summary)

In [ ]:
def process_page(index, use_endpoint=False, endpoint_url=None):
    result = {}
    # OCR text
    ocr_text = df.loc[index, 'ocr_text']
    result['ocr_text'] = ocr_text

    # Embedding
    emb = df.loc[index, 'embedding']
    result['embedding'] = emb

    # Optional Azure score
    if use_endpoint and endpoint_url:
        try:
            payload = {'embedding': emb}
            r = requests.post(endpoint_url, json=payload)
            result['score'] = r.json().get('score')
        except:
            result['score'] = None
    else:
        result['score'] = None

    # GPT‑4o summary
    try:
        resp = client.chat.completions.create(
            model='gpt-4o',
            messages=[
                {'role':'system','content':'You are a financial analyst.'},
                {'role':'user','content': f'Summarise and explain this SEC filing page:
{ocr_text}'}
            ]
        )
        summary = resp.choices[0].message.content
    except Exception as e:
        summary = f'GPT‑4o error: {e}'

    result['gpt4_summary'] = summary
    result['gpt4_explanation'] = summary

    return result

## Run Final Pipeline

In [ ]:
output = process_page(0)
output